# Data Preprocessing

This notebook demonstrates the preprocessing steps used before machine learning.

### Topics covered

1. Load the dataset
2. Select the target column
3. Separate features and target
4. Identify numerical and categorical features
5. Handle missing values
6. Scale numerical features
7. Encode categorical features
8. Build a reusable preprocessing pipeline
9. Transform the dataset
10. Verify the processed data

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split

# Locate the project root automatically
current_path = Path.cwd().resolve()
project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "modules").is_dir():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the AutoML-Studio project root."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import modules.data_loader as data_loader
import modules.preprocessing as preprocessing

print("Project root:", project_root)

C:\Users\PMLS\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
C:\Users\PMLS\AppData\Roaming\Python\Python313\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Project root: C:\Users\PMLS\Documents\Coding\System-7-Internship\AutoML-Studio


In [2]:
sample_path = project_root / "sample_data" / "student-mat.csv"

if not sample_path.exists():
    raise FileNotFoundError(f"Dataset not found: {sample_path}")

df = data_loader.load_csv(sample_path)
data_loader.validate_dataset(df)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
target_column = "G3"

if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' was not found.")

print("Target column:", target_column)
print("\nTarget data type:", df[target_column].dtype)
print("\nTarget values:")
display(df[target_column].value_counts().sort_index())

Target column: G3

Target data type: int64

Target values:


G3
0     38
4      1
5      7
6     15
7      9
8     32
9     28
10    56
11    47
12    31
13    31
14    27
15    33
16    16
17     6
18    12
19     5
20     1
Name: count, dtype: int64

In [4]:
X, y = preprocessing.separate_features_target(df, target_column)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())


Features shape: (395, 32)
Target shape: (395,)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,4,3,4,1,1,3,6,5,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,no,5,3,3,1,1,3,4,5,5
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,no,4,3,2,2,3,3,10,7,8
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,3,2,2,1,1,5,2,15,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,4,3,2,1,2,5,4,6,10


0     6
1     6
2    10
3    15
4    10
Name: G3, dtype: int64

In [5]:
numeric_columns, categorical_columns = preprocessing.identify_column_types(X)

print("Numerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

print("\nNumber of numerical columns:", len(numeric_columns))
print("Number of categorical columns:", len(categorical_columns))

Numerical columns:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Categorical columns:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of numerical columns: 15
Number of categorical columns: 17


In [6]:
missing_features = X.isna().sum()

missing_features = missing_features[
    missing_features > 0
].sort_values(ascending=False)

if not missing_features.empty:
    print("Missing values found in features:")
    display(missing_features.to_frame("Missing Values"))
else:
    print("No missing values found in the feature columns.")

No missing values found in the feature columns.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (316, 32)
Testing features: (79, 32)
Training target: (316,)
Testing target: (79,)


In [8]:
preprocessor = preprocessing.build_preprocessor(
    numeric_columns,
    categorical_columns
)

print("Preprocessing pipeline created successfully.")
display(preprocessor)


Preprocessing pipeline created successfully.


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'Medu', 'Fedu', 'traveltime',
                                  'studytime', 'failures', 'famrel', 'freetime',
                                  'goout', 'Dalc', 'Walc', 'health', 'absences',
                                  'G1', 'G2']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['school', 'sex', 'address', 'famsize',
                                  'Pstatus', 'Mjob', 'Fjob', 'reason',
                                  'guardian', 'schoolsup', 'famsup', 'paid',
                                  'activities', 'nursery', 'higher', 'internet',
                                  'romantic'])])

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data transformed successfully.")
print("Test data transformed successfully.")

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Training data transformed successfully.
Test data transformed successfully.

Processed training shape: (316, 58)
Processed test shape: (79, 58)


In [10]:
print("Processed data type:", type(X_train_processed))
print("Processed training shape:", X_train_processed.shape)

if hasattr(X_train_processed, "toarray"):
    display(
        pd.DataFrame(
            X_train_processed.toarray()
        ).head()
    )
else:
    display(
        pd.DataFrame(X_train_processed).head()
    )

Processed data type: <class 'numpy.ndarray'>
Processed training shape: (316, 58)


,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
0,-0.586396,0.246437,0.423207,-0.625779,-0.056853,-0.456744,0.064432,-1.195981,-0.144396,-0.55443,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
1,-0.586396,-0.680636,0.423207,0.828236,-1.254553,-0.456744,1.195575,-0.214345,-0.144396,-0.55443,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
2,-0.586396,-1.607709,0.423207,-0.625779,-0.056853,3.628097,0.064432,-0.214345,1.644982,-0.55443,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,-0.586396,1.173510,0.423207,-0.625779,1.140847,-0.456744,-1.066710,0.767292,0.750293,0.55443,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
4,-0.586396,-0.680636,-1.434203,0.828236,-1.254553,-0.456744,-1.066710,-0.214345,-1.039085,-0.55443,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


In [11]:
print("Pipeline components:\n")

for name, transformer, columns in preprocessor.transformers:
    print(f"Step: {name}")
    print(f"Columns: {list(columns)}")
    print()

Pipeline components:

Step: numeric
Columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Step: categorical
Columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']



In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nProcessed feature names:")
for feature in feature_names:
    print(feature)

Number of processed features: 58

Processed feature names:
numeric__age
numeric__Medu
numeric__Fedu
numeric__traveltime
numeric__studytime
numeric__failures
numeric__famrel
numeric__freetime
numeric__goout
numeric__Dalc
numeric__Walc
numeric__health
numeric__absences
numeric__G1
numeric__G2
categorical__school_GP
categorical__school_MS
categorical__sex_F
categorical__sex_M
categorical__address_R
categorical__address_U
categorical__famsize_GT3
categorical__famsize_LE3
categorical__Pstatus_A
categorical__Pstatus_T
categorical__Mjob_at_home
categorical__Mjob_health
categorical__Mjob_other
categorical__Mjob_services
categorical__Mjob_teacher
categorical__Fjob_at_home
categorical__Fjob_health
categorical__Fjob_other
categorical__Fjob_services
categorical__Fjob_teacher
categorical__reason_course
categorical__reason_home
categorical__reason_other
categorical__reason_reputation
categorical__guardian_father
categorical__guardian_mother
categorical__guardian_other
categorical__schoolsup_no
categ

In [13]:
if hasattr(X_train_processed, "toarray"):
    processed_array = X_train_processed.toarray()
else:
    processed_array = X_train_processed

processed_df = pd.DataFrame(
    processed_array,
    columns=feature_names
)

print("Missing values after preprocessing:")
print(processed_df.isna().sum().sum())

if processed_df.isna().sum().sum() == 0:
    print("\nNo missing values remain after preprocessing.")
else:
    print("\nMissing values are still present.")

Missing values after preprocessing:
0

No missing values remain after preprocessing.


In [14]:
processed_numeric = processed_df.filter(
    regex=r"^numeric__"
)

if not processed_numeric.empty:
    print("Processed numerical features:")

    display(
        processed_numeric.describe().T[
            ["mean", "std", "min", "max"]
        ]
    )
else:
    print("No processed numerical features found.")

Processed numerical features:


,mean,std,min,max
numeric__age,-6.928354e-16,1.001586,-1.374912,4.144697
numeric__Medu,1.054009e-16,1.001586,-2.534782,1.173510
numeric__Fedu,1.911270e-16,1.001586,-2.362908,1.351912
numeric__traveltime,-1.798842e-16,1.001586,-0.625779,3.736267
numeric__studytime,2.670157e-16,1.001586,-1.254553,2.338548
numeric__failures,3.091760e-17,1.001586,-0.456744,3.628097
numeric__famrel,-1.939377e-16,1.001586,-3.328995,1.195575
numeric__freetime,1.686415e-16,1.001586,-2.177617,1.748928
numeric__goout,3.513364e-17,1.001586,-1.933774,1.644982
numeric__Dalc,0.000000e+00,1.001586,-0.554430,3.881012


The preprocessing workflow completed the following steps:

- Loaded the dataset.
- Selected the target variable.
- Separated features and target.
- Identified numerical and categorical features.
- Split the dataset into training and testing sets.
- Created a reusable `ColumnTransformer`.
- Applied median imputation to numerical features.
- Applied most-frequent imputation to categorical features.
- Applied standard scaling to numerical features.
- Applied one-hot encoding to categorical features.
- Fitted preprocessing only on the training data.
- Transformed both training and testing data.
- Verified the resulting feature matrix.

The resulting `preprocessor` can be combined with a machine learning model inside a `Pipeline`.

This prevents preprocessing steps from being duplicated when training different models.